In [1]:
from pathlib import Path
import pandas as pd
import unicodedata

# Ruta principal del proyecto
PROYECTO = Path.cwd()

# Si el notebook se ejecuta desde la carpeta notebooks,
# retrocedemos a la raiz del repositorio.
if PROYECTO.name == "notebooks":
    PROYECTO = PROYECTO.parent

RUTA_DATOS = (
    PROYECTO
    / "datos"
    / "originales"
    / "informe-de-gastos-e-inversiones.bogota.d.c.2025-12.csv"
)

print("Proyecto:", PROYECTO)
print("Archivo encontrado:", RUTA_DATOS.exists())

Proyecto: c:\Users\acoba\Documents\GitHub\Proyecto-Final-Gestion-Datos-Bogota
Archivo encontrado: True


In [2]:
# 2.1 Leer el archivo original
df_original = pd.read_csv(
    RUTA_DATOS,
    encoding="cp1252",
    sep=None,
    engine="python",
    dtype=str
)

# 2.2 Normalizar encabezados
def normalizar_columna(texto):
    texto = unicodedata.normalize(
        "NFKD", str(texto).strip()
    )
    texto = "".join(
        caracter for caracter in texto
        if not unicodedata.combining(caracter)
    )
    return texto.lower().replace(" ", "_")

df = df_original.copy()
df.columns = [
    normalizar_columna(columna)
    for columna in df.columns
]

# 2.3 Identificar registros presupuestales
df["vigencia"] = df["vigencia"].str.strip()

filas_validas = df["vigencia"].str.fullmatch(
    r"\d{4}-\d{2}",
    na=False
)

filas_excluidas = int((~filas_validas).sum())

df = df.loc[filas_validas].copy()

# 2.4 Limpiar espacios en campos descriptivos
columnas_texto = [
    "tipo_de_entidad",
    "codigo_entidad",
    "nombre_de_la_entidad",
    "codigo_rubro",
    "descripcion_rubro"
]

for columna in columnas_texto:
    df[columna] = df[columna].str.strip()

# 2.5 Mostrar resultados
print("LECTURA Y NORMALIZACION")
print("-" * 45)
print("Registros originales:", len(df_original))
print("Filas informativas excluidas:", filas_excluidas)
print("Registros presupuestales:", len(df))
print("Columnas:", len(df.columns))
print("Vigencias:", df["vigencia"].unique().tolist())

display(df.head())

LECTURA Y NORMALIZACION
---------------------------------------------
Registros originales: 7682
Filas informativas excluidas: 1
Registros presupuestales: 7681
Columnas: 18
Vigencias: ['2025-12']


,vigencia,tipo_de_entidad,codigo_entidad,nombre_de_la_entidad,codigo_rubro,descripcion_rubro,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2025-12,AC,0100,Concejo de Bogotá D.C.,O211010100101,Sueldo básico,3.348.454.408,7.465.785.923,11.245.395.156,15.254.812.867,19.407.384.311,23.547.174.181,27.742.976.432,32.010.985.412,36.387.879.727,40.811.151.814,45.085.284.278,49.604.982.512
1,2025-12,AC,0100,Concejo de Bogotá D.C.,O211010100102,"Horas extras, dominicales, festivos y recargos",123.468.663,248.242.343,373.007.115,491.335.541,642.696.310,772.912.349,915.107.886,1.378.007.910,1.547.900.826,1.701.647.144,1.830.681.047,1.986.961.324
2,2025-12,AC,0100,Concejo de Bogotá D.C.,O211010100103,Gastos de representación,334.565.362,747.693.234,1.120.607.440,1.523.382.309,1.931.291.395,2.329.203.406,2.732.739.638,3.148.618.590,3.571.959.290,3.995.747.207,4.403.700.786,4.834.741.184
3,2025-12,AC,0100,Concejo de Bogotá D.C.,O211010100107,Bonificación por servicios prestados,491.374.428,727.502.371,811.559.472,868.387.828,946.854.551,1.002.813.623,1.093.095.248,1.187.654.586,1.283.257.376,1.409.278.280,1.515.713.489,1.589.969.113
4,2025-12,AC,0100,Concejo de Bogotá D.C.,O21101010010801,Prima de navidad,20.376.516,33.821.643,54.266.371,71.512.289,86.866.868,113.534.992,136.206.619,302.435.927,365.654.533,483.962.212,571.157.207,6.914.675.617


In [3]:
# 3.1 Definir variables mensuales
meses = [
    "enero", "febrero", "marzo", "abril",
    "mayo", "junio", "julio", "agosto",
    "septiembre", "octubre", "noviembre", "diciembre"
]

# 3.2 Analizar completitud y símbolos especiales
resultados_calidad = []

for mes in meses:
    valores = df[mes].astype("string").str.strip()

    vacios = int(
        (valores.isna() | valores.eq("").fillna(False)).sum()
    )

    guiones = int(valores.eq("-").fillna(False).sum())

    valores_numericos = pd.to_numeric(
        valores
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

    no_convertibles = int(
        (
            valores_numericos.isna()
            & valores.notna()
            & valores.ne("").fillna(False)
            & valores.ne("-").fillna(False)
        ).sum()
    )

    resultados_calidad.append({
        "mes": mes,
        "vacios": vacios,
        "guiones": guiones,
        "otros_no_convertibles": no_convertibles,
        "valores_numericos": int(valores_numericos.notna().sum()),
        "porcentaje_sin_valor": round(
            100 * valores_numericos.isna().sum() / len(df), 2
        )
    })

reporte_calidad = pd.DataFrame(resultados_calidad)

print("=== COMPLETITUD Y VALIDEZ MONETARIA ===")
display(reporte_calidad)

# 3.3 Validar unicidad
clave_original = [
    "vigencia", "codigo_entidad", "codigo_rubro"
]

clave_ampliada = [
    "vigencia",
    "codigo_entidad",
    "nombre_de_la_entidad",
    "codigo_rubro"
]

print("\n=== UNICIDAD ===")
print("Duplicados exactos:", df.duplicated().sum())
print(
    "Duplicados con clave original:",
    df.duplicated(subset=clave_original).sum()
)
print(
    "Duplicados con clave ampliada:",
    df.duplicated(subset=clave_ampliada).sum()
)

=== COMPLETITUD Y VALIDEZ MONETARIA ===


,mes,vacios,guiones,otros_no_convertibles,valores_numericos,porcentaje_sin_valor
0,enero,1110,4756,0,1815,76.37
1,febrero,1044,3932,0,2705,64.78
2,marzo,968,3513,0,3200,58.34
3,abril,901,3185,0,3595,53.20
4,mayo,818,2873,0,3990,48.05
5,junio,740,2603,0,4338,43.52
6,julio,611,2466,0,4604,40.06
7,agosto,490,2303,0,4888,36.36
8,septiembre,217,2332,0,5132,33.19
9,octubre,100,2098,0,5483,28.62



=== UNICIDAD ===
Duplicados exactos: 0
Duplicados con clave original: 128
Duplicados con clave ampliada: 0


In [4]:
# 4.1 Crear una copia para el analisis numerico
df_numerico = df.copy()

# 4.2 Convertir los compromisos mensuales a numeros
for mes in meses:
    valores = (
        df_numerico[mes]
        .astype("string")
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    df_numerico[mes] = pd.to_numeric(
        valores,
        errors="coerce"
    )

# 4.3 Calcular diferencias entre meses consecutivos
diferencias = df_numerico[meses].diff(axis=1)

# 4.4 Construir el reporte mensual
resultados_variaciones = []

for i in range(1, len(meses)):
    mes_actual = meses[i]
    mes_anterior = meses[i - 1]

    variacion = diferencias[mes_actual]

    resultados_variaciones.append({
        "mes": mes_actual,
        "mes_anterior": mes_anterior,
        "comparaciones_validas": int(variacion.notna().sum()),
        "disminuciones": int(variacion.lt(0).sum()),
        "aumentos": int(variacion.gt(0).sum()),
        "sin_cambio": int(variacion.eq(0).sum()),
        "monto_disminuciones": variacion[
            variacion.lt(0)
        ].sum()
    })

reporte_variaciones = pd.DataFrame(
    resultados_variaciones
)

print("=== CONSISTENCIA TEMPORAL ===")
display(reporte_variaciones)

# 4.5 Identificar registros con alguna disminucion
filas_con_disminucion = diferencias[meses[1:]].lt(0).any(axis=1)

print("\n=== RESUMEN ===")
print(
    "Registros con al menos una disminucion:",
    int(filas_con_disminucion.sum())
)

print(
    "Porcentaje de registros:",
    round(100 * filas_con_disminucion.mean(), 2),
    "%"
)

# 4.6 Identificar los 10 mayores ajustes negativos
detalle_disminuciones = df_numerico[
    [
        "nombre_de_la_entidad",
        "codigo_rubro",
        "descripcion_rubro"
    ]
].copy()

detalle_disminuciones["mayor_disminucion"] = (
    diferencias[meses[1:]].min(axis=1)
)

mayores_disminuciones = (
    detalle_disminuciones
    .nsmallest(10, "mayor_disminucion")
)

print("\n=== 10 MAYORES DISMINUCIONES ===")
display(mayores_disminuciones)

=== CONSISTENCIA TEMPORAL ===


,mes,mes_anterior,comparaciones_validas,disminuciones,aumentos,sin_cambio,monto_disminuciones
0,febrero,enero,1815,3,1302,510,-87225668
1,marzo,febrero,2705,10,1732,963,-86903514
2,abril,marzo,3200,16,1825,1359,-367293928
3,mayo,abril,3595,29,1848,1718,-4599315136
4,junio,mayo,3990,31,1857,2102,-4981859726
5,julio,junio,4338,60,1912,2366,-4024580085
6,agosto,julio,4604,50,1989,2565,-55697040562
7,septiembre,agosto,4888,59,2014,2815,-17714138074
8,octubre,septiembre,5131,100,2167,2864,-12802892348
9,noviembre,octubre,5482,89,2060,3333,-11390640089



=== RESUMEN ===
Registros con al menos una disminucion: 669
Porcentaje de registros: 8.71 %

=== 10 MAYORES DISMINUCIONES ===


,nombre_de_la_entidad,codigo_rubro,descripcion_rubro,mayor_disminucion
1517,Secretaría de Educación del Distrito,O232020200883325,Servicios de ingeniería en proyectos de teleco...,-41432048128
2713,Secretaría Distrital de Integración Social,O232020200991199,Otros servicios administrativos del gobierno n...,-11402129194
7498,Agencia Distrital para la Educación Superior l...,O232020200992512,Servicios de educación superior nivel pregrado...,-11370907373
7220,Unidad Administrativa Especial de Servicios Pú...,O2130509059,Servicio de alumbrado público,-6219481407
4093,Fondo Financiero de Salud - FFDS,O23201010010208,Edificios relacionados con salud,-4189123279
3666,"Secretaría Distrital de Seguridad, Convivencia...",O232020200663393,Otros servicios de comidas contratadas,-3689167510
1521,Secretaría de Educación del Distrito,O232020200885330,Servicios de limpieza general,-2967187739
7497,Agencia Distrital para la Educación Superior l...,O232020200992511,Servicios de educación superior nivel pregrado...,-2513073653
2036,Secretaría Distrital del Hábitat,O2320202005040254211,Servicios generales de construcción de carrete...,-2297919261
1520,Secretaría de Educación del Distrito,O232020200885250,Servicios de protección (guardas de seguridad),-1098333962


In [5]:
# 5.1 Crear directorio de salida
RUTA_SALIDA = PROYECTO / "datos" / "procesados"
RUTA_SALIDA.mkdir(parents=True, exist_ok=True)

# 5.2 Crear identificador tecnico por registro
# No reemplaza la clave de negocio validada.
df_exportacion = df_numerico.copy().reset_index(drop=True)
df_exportacion.insert(
    0,
    "linea_id",
    range(1, len(df_exportacion) + 1)
)

# 5.3 Exportar datos normalizados
df_exportacion.to_csv(
    RUTA_SALIDA / "compromisos_2025_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5.4 Transformar de formato ancho a largo
identificadores = [
    "linea_id",
    "vigencia",
    "tipo_de_entidad",
    "codigo_entidad",
    "nombre_de_la_entidad",
    "codigo_rubro",
    "descripcion_rubro"
]

df_mensual = df_exportacion.melt(
    id_vars=identificadores,
    value_vars=meses,
    var_name="nombre_mes",
    value_name="compromiso_acumulado"
)

# 5.5 Incorporar numero de mes y fecha
mapa_meses = {
    mes: numero
    for numero, mes in enumerate(meses, start=1)
}

df_mensual["mes"] = (
    df_mensual["nombre_mes"].map(mapa_meses)
)

df_mensual["fecha"] = pd.to_datetime(
    df_mensual["vigencia"].str[:4]
    + "-"
    + df_mensual["mes"].astype(str).str.zfill(2)
    + "-01"
)

# 5.6 Calcular variacion mensual por registro
df_mensual = df_mensual.sort_values(
    ["linea_id", "mes"]
)

df_mensual["compromiso_mes"] = (
    df_mensual
    .groupby("linea_id")["compromiso_acumulado"]
    .diff()
)

# 5.7 Exportar tabla mensual
df_mensual.to_csv(
    RUTA_SALIDA / "compromisos_2025_mensual.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5.8 Exportar reportes de calidad
reporte_calidad.to_csv(
    RUTA_SALIDA / "reporte_calidad_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

reporte_variaciones.to_csv(
    RUTA_SALIDA / "reporte_variaciones_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

# 5.9 Validaciones de exportacion
assert len(df_exportacion) == 7681
assert len(df_mensual) == len(df_exportacion) * 12
assert df_exportacion[clave_ampliada].duplicated().sum() == 0

print("=== EXPORTACION FINALIZADA ===")
print("Directorio:", RUTA_SALIDA)
print("Registros limpios:", len(df_exportacion))
print("Registros mensuales:", len(df_mensual))
print("Archivos generados:")

for archivo in sorted(RUTA_SALIDA.glob("*.csv")):
    print("-", archivo.name)

=== EXPORTACION FINALIZADA ===
Directorio: c:\Users\acoba\Documents\GitHub\Proyecto-Final-Gestion-Datos-Bogota\datos\procesados
Registros limpios: 7681
Registros mensuales: 92172
Archivos generados:
- compromisos_2025_limpio.csv
- compromisos_2025_mensual.csv
- reporte_calidad_2025.csv
- reporte_variaciones_2025.csv


In [6]:
# 6.1 Leer los archivos procesados
ruta_limpio = RUTA_SALIDA / "compromisos_2025_limpio.csv"
ruta_mensual = RUTA_SALIDA / "compromisos_2025_mensual.csv"

df_limpio_check = pd.read_csv(
    ruta_limpio,
    encoding="utf-8-sig",
    dtype={"codigo_entidad": str, "codigo_rubro": str}
)

df_mensual_check = pd.read_csv(
    ruta_mensual,
    encoding="utf-8-sig",
    dtype={"codigo_entidad": str, "codigo_rubro": str}
)

# 6.2 Verificar cantidad de registros
assert len(df_limpio_check) == len(df_exportacion)
assert len(df_mensual_check) == len(df_exportacion) * 12

# 6.3 Verificar identificadores
assert df_limpio_check["linea_id"].is_unique
assert df_mensual_check.groupby("linea_id").size().eq(12).all()

# 6.4 Comparar importes de diciembre
total_diciembre_ancho = df_limpio_check["diciembre"].sum()

total_diciembre_largo = df_mensual_check.loc[
    df_mensual_check["mes"] == 12,
    "compromiso_acumulado"
].sum()

assert total_diciembre_ancho == total_diciembre_largo

# 6.5 Verificar que las variaciones coincidan
disminuciones_exportadas = int(
    df_mensual_check["compromiso_mes"].lt(0).sum()
)

disminuciones_originales = int(
    reporte_variaciones["disminuciones"].sum()
)

assert disminuciones_exportadas == disminuciones_originales

# 6.6 Mostrar resultados
print("=== VALIDACION DE INTEGRIDAD ===")
print("Registros limpios:", len(df_limpio_check))
print("Registros mensuales:", len(df_mensual_check))
print("Identificadores unicos:", df_limpio_check["linea_id"].nunique())
print("Total diciembre ancho:", f"{total_diciembre_ancho:,.0f}")
print("Total diciembre largo:", f"{total_diciembre_largo:,.0f}")
print("Disminuciones mensuales:", disminuciones_exportadas)
print("Estado: VALIDACIONES SUPERADAS")

=== VALIDACION DE INTEGRIDAD ===
Registros limpios: 7681
Registros mensuales: 92172
Identificadores unicos: 7681
Total diciembre ancho: 36,256,796,365,772
Total diciembre largo: 36,256,796,365,772
Disminuciones mensuales: 838
Estado: VALIDACIONES SUPERADAS
